# Federated unlearning sweep (Kaggle)

Federated counterpart of the single-node sweep: a global model is trained by FedAvg over several clients (client 0 holds the forget set), then that client is unlearned and attacked. Reports residual leakage, the outlier/typical equity gap, and the **update-signal AUC** (whether the removal update itself reveals the forgotten records).

**Before running:** push the federated code to GitHub (`git add -A && git commit -m 'federated' && git push`) so the clone below includes `federated.py`, or upload the repo as a Kaggle dataset. Turn on the GPU and Internet in the notebook settings.

In [ ]:
!git clone -q https://github.com/Thommartial/llm-unlearning-experiments.git
%cd llm-unlearning-experiments
!pip -q install -r requirements.txt && pip -q install -e .
!python scripts/verify_gpu.py

In [ ]:
# Validate the pipeline offline in seconds before spending GPU time.
!python scripts/run_federated.py --config configs/smoke_fed_synthetic.yaml --run

In [ ]:
# Federated TOFU sweep. Resumable: finished (model, seed) pairs are skipped,
# so you can survive Kaggle session limits by re-running this cell.
import sys, itertools, json
sys.path.insert(0, "src")  # editable install isn't visible to the already-running kernel
from pathlib import Path
import pandas as pd
from unlearning.config import load_config
from unlearning.federated import run_federated

models = ["distilgpt2", "gpt2"]         # add "gpt2-medium" on the 16 GB card
seeds = [0, 1, 2]
rows = []
for model, seed in itertools.product(models, seeds):
    cfg = load_config("configs/fed_tofu.yaml")
    cfg.model.name = model
    cfg.seed = seed
    cfg.name = f"fed_tofu_{model}_seed{seed}"
    out = Path(cfg.output_dir) / cfg.name
    mfile = out / "metrics.json"
    m = json.loads(mfile.read_text()) if mfile.exists() else run_federated(cfg, out)
    post = m["post_unlearning"]
    rows.append({
        "model": model, "seed": seed,
        "pre_auc": m["pre_unlearning"]["mia_auc"],
        "post_auc": post["mia_auc"],
        "post_adv_outlier": post.get("mia_advantage_outlier"),
        "post_adv_typical": post.get("mia_advantage_typical"),
        "update_signal_auc": m["update_signal_auc"],
        "forget_loss_pre": m["forget_loss_pre"], "forget_loss_post": m["forget_loss_post"],
        "retain_loss_pre": m["retain_loss_pre"], "retain_loss_post": m["retain_loss_post"],
    })
Path("results").mkdir(exist_ok=True)
df = pd.DataFrame(rows)
df.to_csv("results/fed_sweep.csv", index=False)
df

Download `results/fed_sweep.csv` and each `results/fed_tofu_*/metrics.json` from the Kaggle output panel. The key columns for the paper: `post_auc` (residual leakage after federated unlearning), `post_adv_outlier` vs `post_adv_typical` (equity gap), and `update_signal_auc` (the second-order removal signal).